# v7: extend P=10 fine-tuning from v6

This continues each of the three v6 P=10 checkpoints for up to 100 additional epochs, starting at a lower learning rate with automatic reduction when validation stalls. It selects by the worst per-SNR validation DL-minus-GS gap. The test split is evaluated only after selection. Choose a T4 GPU before running the training cell.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os, torch
root = '/content/gdrive/MyDrive'
required = [
    f'{root}/DL_dataset_RSR25dB/{split}_P10.mat'
    for split in ('train', 'validation', 'test')
] + [
    f'{root}/DL_dataset_RSR25dB/v6_starting_models/model_P10_seed{seed}.pt'
    for seed in (20261026, 20261027, 20261028)
] + [
    f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_{split}_P10.json'
    for split in ('validation', 'test')
]
missing = [p for p in required if not os.path.isfile(p)]
assert not missing, f'Missing Drive files: {missing}'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('Dataset, v6 models, and GS curves found.')

In [ ]:
from google.colab import files
uploaded = files.upload()
required_scripts = {
    'colab_unfolded_dl_v3_highsnr.py',
    'colab_unfolded_dl_v5_snr_aware.py',
    'colab_unfolded_dl_v6_gs_gap.py',
    'colab_unfolded_dl_v7_continue_p10.py',
}
missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All four scripts uploaded.')

In [ ]:
assert torch.cuda.is_available(), 'Select a T4 GPU runtime before training.'
!python /content/colab_unfolded_dl_v7_continue_p10.py

In [ ]:
import json
result_dir = '/content/gdrive/MyDrive/unfolded_dl_v7_p10_extended_results'
with open(f'{result_dir}/summary_unfolded_v7_p10_extended.json') as f:
    results = json.load(f)
for result in results:
    print(f"seed={result['seed']} additional_best_epoch={result['best_additional_epoch']}")
    for snr, gap in result['test_dl_minus_gs_db_by_snr'].items():
        print(f'  {snr:>3} dB: DL minus GS = {gap:+.4f} dB')
